## Required qualifier Repair Analyzer

In [ ]:
import pandas as pd

# You can read the already calculated repairs file, or read the one generated on 't-box repairs analyzer' folder with the script 't-box_repairs_analyzer.py'
df_required = pd.read_csv("../../required_qualifier_repairs.csv")

df_required

- The cell below counts different types of basic T-box repairs generated with the relational database:

In [ ]:
print(len(df_required[(df_required['C_deleted'] == True)] ))
print(len(df_required[(df_required['C_deprecated'] == True)] ))
print(len(df_required[(df_required['CQ_added_exception'] == True)] ))
print(len(df_required[(df_required['CQ_removed_property'] == True)] ))



- check for base statement deletions:

In [ ]:
import requests
import xml.etree.ElementTree as ET

# Function to check if a statement has been removed
def isRemoved(subject, p_prop, stmt):
    endpoint = "ENTER_qEndpoint_WD_2023"

    query = f"""ASK {{ <{subject}><{p_prop}><{stmt}> }}"""
    encoded_query = requests.utils.quote(query)
    url = f"{endpoint}?query={encoded_query}"
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    # Retry logic
    attempts = 3
    for attempt in range(attempts):
        try:
            response = requests.get(url, headers=headers)
            if response.ok:
                root = ET.fromstring(response.text)
                boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
                if boolean_element is not None:
                    return boolean_element.text.lower() == 'false'
            else:
                print(f"Request failed: {response.status_code} - {response.text}")
        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
        
        # If not successful, wait before retrying
        if attempt < attempts - 1:
            print("Retrying...")
            
    # If all attempts fail, return None
    return None

# Example usage
subject = "http://www.wikidata.org/entity/Q2919989"
wd_p = "http://www.wikidata.org/prop/P39"
stmt = "http://www.wikidata.org/entity/statement/Q2919989-7E2360D1-2FDA-48ED-99F1-500CD9D87EC0"
print(isRemoved(subject,wd_p, stmt))


In [ ]:
df_required['S_deleted'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Load the data, or resume from the last checkpoint
checkpoint_file = "checkpoint.csv"
if os.path.exists(checkpoint_file):
    df_required = pd.read_csv(checkpoint_file)
    print("Resuming from the last checkpoint.")
else:
    print("Starting from scratch.")

# Process the rows with progress bar
for index, row in tqdm(df_required.iterrows(), total=len(df_required)):
    # Check for unprocessed rows
    if pd.isna(row['S_deleted']):
        result = isRemoved(row['subject'], row['p_pid'], row['obj_statement'])
        df_required.at[index, 'S_deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 10000 == 0:
        df_required.to_csv(checkpoint_file, index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_required.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_required[(df_required['S_deleted'] == True)] )

- checking rows still to be classified:

In [ ]:
df_required[(df_required['C_deleted'] == False) & 
     (df_required['C_deprecated'] == False)& 
     (df_required['CQ_added_exception'] == False)& 
     (df_required['CQ_removed_property'] == False) & 
     (df_required['S_deleted'] == False) 
     
    ]

- test for A-box additions of already existent required qualifiers (SQ+):

In [ ]:
df_required["SQ_added_property"] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def reqQualifierAdded(obj_statement, pq_qualifier_must_be_used):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{obj_statement}><{pq_qualifier_must_be_used}>[]    }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
obj_statement = "http://www.wikidata.org/entity/statement/Q83359-9DFD01B6-DC0A-4198-A6D4-8E9D41BA0995"
pq_qualifier_must_be_used = "http://www.wikidata.org/prop/qualifier/P585"
print(reqQualifierAdded(obj_statement,pq_qualifier_must_be_used))


In [ ]:
df_required.to_csv("required_qualifier_repairs.csv", index=False)

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Load the data, or resume from the last checkpoint
checkpoint_file = "checkpoint.csv"
if os.path.exists(checkpoint_file):
    df_required = pd.read_csv(checkpoint_file)
    print("Resuming from the last checkpoint.")
else:
    print("Starting from scratch.")

# Process the rows with progress bar
for index, row in tqdm(df_required.iterrows(), total=len(df_required)):
    
    # Check for unprocessed rows
    if pd.isna(row['SQ_added_property']):
        result = reqQualifierAdded(row['obj_statement'], row['pq_qualifier_must_be_used'])
        df_required.at[index, 'SQ_added_property'] = result

    # Save a checkpoint every 10,000 rows
    if index % 10000 == 0:
        df_required.to_csv(checkpoint_file, index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_required.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_required[(df_required['C_deleted'] == False) & 
     (df_required['C_deprecated'] == False)& 
     (df_required['CQ_added_exception'] == False)& 
     (df_required['CQ_removed_property'] == False) & 
     (df_required['S_deleted'] == False) & 
     (df_required['SQ_added_property'] == False) 
     
    ]

In [ ]:
len(df_required[(df_required['SQ_added_property'] == True)] )

In [ ]:
df_required.dtypes

In [ ]:
df_required['SQ_added_property'] = df_required['SQ_added_property'].astype(bool)
df_required.to_csv("required_qualifier_final_repairs.csv",index=False)

- generate Venn diagram with repair shares:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# Create the new DataFrame with the required columns
df2 = pd.DataFrame()
df2['A-box changes'] = df_required['S_deleted'] | df_required['SQ_added_property']
df2['T-box changes'] = (
    df_required['C_deleted'] | 
    df_required['C_deprecated'] | 
    df_required['CQ_added_exception'] | 
    df_required['CQ_removed_property']
)

# Calculate the sizes of the sets
a_box_changes = df2['A-box changes'].sum()
t_box_changes = df2['T-box changes'].sum()
intersection = (df2['A-box changes'] & df2['T-box changes']).sum()

# Plot the Venn diagram
venn2(subsets=(a_box_changes, t_box_changes, intersection), 
      set_labels=('A-box changes', 'T-box changes'))


# Add title
plt.title("Required Qualifiers Constraint repairs")

plt.show()
